In [16]:
import pandas as pd
import numpy as np

# ⚡ Vectorized String Operations in Pandas

**Topic:** 07.07 · **Level:** 🟠 Intermediate · **Type:** THEORY + CODE

## 📖 Vectorized operations

```python
a = np.array([1, 2, 3, 4])
a * 4                    # elementwise → [4,8,12,16]
```

- Vectorized = ek operation har element pe, loop nahi
- Numpy/pandas core speed
- Strings pe direct Python ops me None/NaN fail hote hain

## 🔬 Deep Dive : Vectorized ops — pandas solves python's string slowness

```python
a = np.array([1,2,3,4]); a * 4        # fast elementwise — vectorized

s = ['cat','mat',None,'rat','sitting','batting']     # plain python list
# [x.upper() for x in s]  -> None.upper() crashes TypeError!
```

- Vectorized: op applied to WHOLE array at once (compiled C)
- Python loop: per-element, slow + error-prone
- **Pandas `.str`**: missing values (None/NaN) auto-ignored → no crash
```python
s2 = pd.Series(['cat',None,'rat'])
s2.str.upper()    # NaN stays NaN, rest uppercase
```
- .str = vectorized string methods on Series
- Chaining: s.str.lower().str.strip()...
- NaN propagation handled for you
- This is why pandas over raw python for text columns

In [18]:
# What are vectorized operations
a = np.array([1,2,3,4])
a * 4

array([ 4,  8, 12, 16])

In [19]:
# problem in vectorized opertions in vanilla python
s = ['cat','mat',None,'rat']

[i.startswith('c') for i in s]

AttributeError: ignored

## 📖 .str accessor

```python
s = pd.Series(['cat', 'mat', None, 'rat'])
s.str.upper()    # nan/None handled automatically
```

- `.str` — string methods access karta hai vectorized
- Missing values (None/NaN) wapas skip ho jate hain
- Python loops se kaafi fast aur safe
- `.str` ke saath har string method chalega

In [21]:
# How pandas solves this issue?

s = pd.Series(['cat','mat',None,'rat'])
# string accessor
s.str.startswith('c')

# fast and optimized

0     True
1    False
2     None
3    False
dtype: object

In [23]:
# import titanic
df = pd.read_csv('/content/titanic.csv')
df['Name']

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
                             ...                        
886                                Montvila, Rev. Juozas
887                         Graham, Miss. Margaret Edith
888             Johnston, Miss. Catherine Helen "Carrie"
889                                Behr, Mr. Karl Howell
890                                  Dooley, Mr. Patrick
Name: Name, Length: 891, dtype: object

## 📖 Common string methods

```python
df['Name'].str.upper()                     # case
df['Name'].str.lower()
df['Name'].str.capitalize()
df['Name'].str.title()
df['lastname'] = df['Name'].str.split(',').str.get(0)  # split
```

- `lower/upper/capitalize/title` — case uniform
- `split(',')` → list; `.str.get(i)` → element
- `.str.strip()` — whitespace hatao
- Chain: `split → get → strip → split`

## 🔬 Deep Dive : Common string methods

```python
df['Name'].str.upper() / .lower() / .title() / .capitalize()
df['lastname'] = df['Name'].str.split(',').str.get(0)
df[['title','firstname']] = df['Name'].str.split(',') \
                                .str.get(1).str.strip() \
                                .str.split(' ', n=1, expand=True)
```

- case: upper/lower/title/capitalize — uniform text
- split(',') → Series of LISTS
- .str.get(i) → pick i-th element
- .str.strip() — remove leading/trailing spaces
- split with expand=True + n=1 → DataFrame multiple cols
- chain accessors — pipeline one-shot
- .str.len(), .str.contains('x'), .str.replace()
- titanic Name parsing: lastname/title/firstname extraction
- All vectorized, handle NaN
- Cleaning: normalize whitespace & case before matching

In [35]:
# Common Functions
# lower/upper/capitalize/title
df['Name'].str.upper()
df['Name'].str.capitalize()
df['Name'].str.title()
# len
df['Name'][df['Name'].str.len() == 82].values[0]
# strip
"                   nitish                              ".strip()
df['Name'].str.strip()

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
                             ...                        
886                                Montvila, Rev. Juozas
887                         Graham, Miss. Margaret Edith
888             Johnston, Miss. Catherine Helen "Carrie"
889                                Behr, Mr. Karl Howell
890                                  Dooley, Mr. Patrick
Name: Name, Length: 891, dtype: object

In [41]:
# split -> get
df['lastname'] = df['Name'].str.split(',').str.get(0)
df.head()

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked   lastname  
0      0         A/5 21171   7.2500   NaN        S     Braund  
1      0          PC 17599  71.2833   C85        C    Cumings  
2      0  STON/O2. 3101282   7.9250   NaN        S  Heikkinen  
3      0            113803  53.1000  C123        S   Futrelle  


In [59]:
df[['title','firstname']] = df['Name'].str.split(',').str.get(1).str.strip().str.split(' ', n=1, expand=True)
df.head()

df['title'].value_counts()

Mr.          517
Miss.        182
Mrs.         125
Master.       40
Dr.            7
Rev.           6
Mlle.          2
Major.         2
Col.           2
the            1
Capt.          1
Ms.            1
Sir.           1
Lady.          1
Mme.           1
Don.           1
Jonkheer.      1
Name: title, dtype: int64

## 📖 replace & filtering

```python
df['title'] = df['title'].str.replace('Ms.', 'Miss.')
df[df['firstname'].str.endswith('A')]       # filter rows
df[df['firstname'].str.startswith(...)]
df['Name'].str[::-1]                         # reverse slice
```

- `replace(old, new)` — substring replace
- `startswith/endswith` — boolean filter
- `contains(regex)` — pattern search (regex support)
- String slicing `[::-1]` bhi kaam karta hai

## 🔬 Deep Dive : replace & filtering strings

```python
df['title'] = df['title'].str.replace('Ms.','Miss.')   # exact string swap
df[df['firstname'].str.endswith('A')]          # filter rows
df[df['firstname'].str.startswith('William')]  # prefix rows
```

- replace(old, new) — string substitute (exact substring by default, regex=True option)
- startswith / endswith — prefix/suffix filter → bool mask → df[mask]
- combine: df[df['name'].str.contains('Mr.', na=False)]
- `na=False` keeps rows where NaN doesn't get dropped by comparison
- filter = the main string use in analysis (name, city, id patterns)
- pattern matching both edge-based and contains-based
- output keeps full df, only filters rows
- value_counts on cleaned titles — insight after cleanup
- boolean masks compose with & | ~

In [60]:
# replace
df['title'] = df['title'].str.replace('Ms.','Miss.')
df['title'] = df['title'].str.replace('Mlle.','Miss.')

<ipython-input-60-9403e5934d1f>:2: FutureWarning: The default value of regex will change from True to False in a future version.
  df['title'] = df['title'].str.replace('Ms.','Miss.')
<ipython-input-60-9403e5934d1f>:3: FutureWarning: The default value of regex will change from True to False in a future version.
  df['title'] = df['title'].str.replace('Mlle.','Miss.')


In [61]:
df['title'].value_counts()

Mr.          517
Miss.        185
Mrs.         125
Master.       40
Dr.            7
Rev.           6
Major.         2
Col.           2
Don.           1
Mme.           1
Lady.          1
Sir.           1
Capt.          1
the            1
Jonkheer.      1
Name: title, dtype: int64

In [66]:
# filtering
# startswith/endswith
df[df['firstname'].str.endswith('A')]
# isdigit/isalpha...
df[df['firstname'].str.isdigit()]

Empty DataFrame
Columns: [PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, lastname, title, firstname]
Index: []

## 📖 Regex with .str

```python
df[df['firstname'].str.contains('John', case=False)]
```

- `contains` — regex pattern match
- `case=False` — case-insensitive
- Filtering = boolean mask
- Data cleaning me common (irregular text)

## 🔬 Deep Dive : Regex with .str

```python
# contains -> search substring or pattern
df[df['firstname'].str.contains('john', regex=True)]              # case sensitive
df[df['firstname'].str.contains('john|John', regex=True)]         # alternation
df[df['firstname'].str.contains('.*john.*', flags=re.I, regex=True)]  # ignorecase whole string
df['Name'].str[::-1]     # reverse each name (slicing on strings!)
```

- str.contains(pattern, regex=True) — substring or regex test
- flags=re.I — case-insensitive
- alternation |, wildcards ., anchors ^ $, [a-z] classes
- str[::-1] — python string slicing applied elementwise
- str.findall/extract for capture groups (complex)
- regex = pattern languages for text matching
- Filter by patterns — phone, email, code columns
- escape specials, test carefully
- regex slows down — simple substring first via regex=False
- Every text col query pattern starts here

In [72]:
# applying regex
# contains
# search john -> both case
df[df['firstname'].str.contains('john',case=False)]
# find lastnames with start and end char vowel
df[df['lastname'].str.contains('^[^aeiouAEIOU].+[^aeiouAEIOU]$')]

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
5              6         0       3   
6              7         0       1   
..           ...       ...     ...   
884          885         0       3   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
5                                     Moran, Mr. James    male   NaN      0   
6                              McCarthy, Mr. Timothy J    male  54.0      0   
..                                                 ...     ...   ... 

In [76]:
# slicing
df['Name'].str[::-1]

0                                sirraH newO .rM ,dnuarB
1      )reyahT sggirB ecnerolF( yeldarB nhoJ .srM ,sg...
2                                 aniaL .ssiM ,nenikkieH
3           )leeP yaM yliL( htaeH seuqcaJ .srM ,ellertuF
4                               yrneH mailliW .rM ,nellA
                             ...                        
886                                sazouJ .veR ,alivtnoM
887                         htidE teragraM .ssiM ,maharG
888             "eirraC" neleH enirehtaC .ssiM ,notsnhoJ
889                                llewoH lraK .rM ,rheB
890                                  kcirtaP .rM ,yelooD
Name: Name, Length: 891, dtype: object